In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
try:
    import plotly.express as px  # optional: only commented interactive cells use it
except ModuleNotFoundError:
    px = None
from matplotlib import colors
import joypy

In [ ]:
import os
from repro_helpers import REPO_ROOT, REPO_DATA

# Portable stand-in for the original cluster `base_path`. The notebook writes any
# figures under ./_generated (gitignored); all data is read from the repo via
# repro_helpers, which replaces the cluster-only src.preprocess / src.pair_table_global.
base_path = str(REPO_ROOT / "src" / "graph_analysis" / "_generated")
os.makedirs(f"{base_path}/data/outputs/figures/summary-stats", exist_ok=True)

In [ ]:
from repro_helpers import preprocess_data

In [ ]:
def set_font_style():
    # Font settings following Nature journal guidelines
    plt.rcParams['font.family'] = 'sans-serif'
    plt.rcParams['font.sans-serif'] = ['DejaVu Sans']  # Default to a reliable system font
    
    # Nature-compliant figure styling
    plt.rcParams['font.size'] = 8        # Base font size
    plt.rcParams['axes.titlesize'] = 8   # Title size
    plt.rcParams['axes.labelsize'] = 8   # Axis label size
    plt.rcParams['xtick.labelsize'] = 6  # Tick label size
    plt.rcParams['ytick.labelsize'] = 6  # Tick label size

# Figure size options (in inches)
# Adjusted based on Nature examples, emphasizing horizontal layouts
FIGURE_SIZES = {
    "single_column": (3.5, 2.5),  # Single-column figure
    "double_column": (7.2, 4.5),  # Double-column figure
    "panel_figure": (7.2, 6.0),   # Multi-panel figures
    "full_page": (7.2, 9.0),      # Full-page figure
}

In [ ]:
wildtype = "............."
dead_mutant = "XXXXXXXXXXXXX"

In [ ]:
# File paths
amp_path = f"{base_path}/data/raw/combined-auc/genotype_auc_sorted_ampicillin.csv"
azt_path = f"{base_path}/data/raw/combined-auc/genotype_auc_sorted_aztreonam.csv"
amp_pairs_path = f"{base_path}/data/processed/amp_pairs.csv"
azt_pairs_path = f"{base_path}/data/processed/azt_pairs.csv"

# Load and preprocess data
processed_data = preprocess_data(amp_path, azt_path, amp_pairs_path, azt_pairs_path, clean_nulls_flag=True)

# Access the processed dataframes
amp_df = processed_data['amp']['original']
amp_long_df = processed_data['amp']['long']
amp_pairs_df = processed_data['amp']['pairs']

azt_df = processed_data['azt']['original']
azt_long_df = processed_data['azt']['long']
azt_pairs_df = processed_data['azt']['pairs']


In [ ]:
def split_mutant(mutant):
    mid = len(mutant) // 2
    return mutant[:mid], mutant[mid:]

# def calculate_statistics(df):
#     rep1 = df['replicate1'].to_numpy()
#     rep2 = df['replicate2'].to_numpy()
#     rep3 = df['replicate3'].to_numpy()
#     rep_arr = np.stack([rep1, rep2, rep3], axis=1)

#     # convert back to exponential scale
#     rep_arr = np.power(10, rep_arr)

#     mean = np.nanmean(rep_arr, axis=1)
#     median = np.nanmedian(rep_arr, axis=1)
#     std = np.nanstd(rep_arr, axis=1)
    
#     # Handle division by zero for cv_std
#     cv_std = np.zeros_like(std)
#     nonzero_mean = mean != 0
#     cv_std[nonzero_mean] = std[nonzero_mean] / mean[nonzero_mean]
    
#     # Handle division by zero for z_scores
#     z_scores = np.zeros_like(rep_arr)
#     nonzero_std = std != 0
#     z_scores[nonzero_std] = (rep_arr[nonzero_std] - mean[nonzero_std, None]) / std[nonzero_std, None]
    
#     z_median = np.nanmedian(z_scores, axis=1)

#     mutant_profiles = df['mutant_profile'].to_numpy()
#     mutant_part1, mutant_part2 = zip(*[split_mutant(mutant) for mutant in mutant_profiles])

#     stats_df = pl.DataFrame({
#         'mutant_profile': df['mutant_profile'],
#         'concentration': df['concentration'],
#         'mutant_part1': mutant_part1,
#         'mutant_part2': mutant_part2,
#         'replicate1': df['replicate1'],
#         'replicate2': df['replicate2'],
#         'replicate3': df['replicate3'],
#         'mean': np.log10(mean),
#         'median': np.log10(median),
#         'std': np.log10(std),
#         'cv_std': np.log10(cv_std),
#         'z_median': np.log10(z_median),
#         'z_scores1': z_scores[:, 0],
#         'z_scores2': z_scores[:, 1],
#         'z_scores3': z_scores[:, 2]
#     })

#     return stats_df

def calculate_statistics(df):
    rep1 = df['replicate1'].to_numpy()
    rep2 = df['replicate2'].to_numpy()
    rep3 = df['replicate3'].to_numpy()
    rep_arr = np.stack([rep1, rep2, rep3], axis=1)

    # Calculate statistics directly in log10 space
    mean = np.nanmean(rep_arr, axis=1)
    median = np.nanmedian(rep_arr, axis=1)
    std = np.nanstd(rep_arr, axis=1)
    
    # Handle division by zero for cv_std
    cv_std = np.zeros_like(std)
    nonzero_mean = mean != 0
    cv_std[nonzero_mean] = std[nonzero_mean] / mean[nonzero_mean]
    
    # Handle division by zero for z_scores
    z_scores = np.zeros_like(rep_arr)
    nonzero_std = std != 0
    z_scores[nonzero_std] = (rep_arr[nonzero_std] - mean[nonzero_std, None]) / std[nonzero_std, None]
    
    z_median = np.nanmedian(z_scores, axis=1)

    mutant_profiles = df['mutant_profile'].to_numpy()
    mutant_part1, mutant_part2 = zip(*[split_mutant(mutant) for mutant in mutant_profiles])

    stats_df = pl.DataFrame({
        'mutant_profile': df['mutant_profile'],
        'concentration': df['concentration'],
        'mutant_part1': mutant_part1,
        'mutant_part2': mutant_part2,
        'replicate1': df['replicate1'],
        'replicate2': df['replicate2'],
        'replicate3': df['replicate3'],
        'mean': mean,
        'median': median,
        'std': std,
        'cv_std': cv_std,
        'z_median': z_median,
        'z_scores1': z_scores[:, 0],
        'z_scores2': z_scores[:, 1],
        'z_scores3': z_scores[:, 2]
    })

    return stats_df

def substract_wildtype(df, wildtype="............."):
    # Filter wildtype data
    wildtype_df = df.filter(pl.col("mutant_profile") == wildtype)

    # Create dictionaries for wildtype mean and median values
    wildtype_mean_dict = dict(zip(wildtype_df["concentration"], wildtype_df["mean"]))
    wildtype_median_dict = dict(zip(wildtype_df["concentration"], wildtype_df["median"]))

    # Create new columns with explicit return dtype
    df = df.with_columns([
        (pl.col("mean") - pl.col("concentration").map_elements(
            lambda x: wildtype_mean_dict.get(x), 
            return_dtype=pl.Float64
        )).alias("mean_w"),
        (pl.col("median") - pl.col("concentration").map_elements(
            lambda x: wildtype_median_dict.get(x),
            return_dtype=pl.Float64
        )).alias("median_w")
    ])

    return df

In [ ]:
amp_stats_df = calculate_statistics(amp_long_df)
azt_stats_df = calculate_statistics(azt_long_df)

amp_stats_df = substract_wildtype(amp_stats_df)
azt_stats_df = substract_wildtype(azt_stats_df)

In [ ]:
print("Long DataFrame columns:")
print(amp_long_df.columns)
print("\nStats DataFrame columns:")
print(amp_stats_df.columns)


In [ ]:
amp_stats_df = amp_stats_df.sort("mutant_profile")
azt_stats_df = azt_stats_df.sort("mutant_profile")

In [ ]:
from repro_helpers import calculate_normalized_fitness

In [ ]:
amp_global_fitness_df = calculate_normalized_fitness(amp_long_df)
azt_global_fitness_df = calculate_normalized_fitness(azt_long_df)

In [ ]:
amp_global_fitness_df.sort("mutant_profile")

In [ ]:
# # Save the sorted global fitness dataframes to CSV files
# amp_global_fitness_df.sort("mutant_profile").write_csv("/work/greencenter/s439821/TEM1CML/data/processed/amp_global_fitness.csv")
# azt_global_fitness_df.sort("mutant_profile").write_csv("/work/greencenter/s439821/TEM1CML/data/processed/azt_global_fitness.csv")

# print("Global fitness dataframes saved to CSV files.")


In [ ]:
# Add mutant parts columns to global fitness dataframes
amp_global_fitness_df = amp_global_fitness_df.with_columns([
    pl.col('mutant_profile').map_elements(lambda x: split_mutant(x)[0], return_dtype=str).alias('mutant_part1'),
    pl.col('mutant_profile').map_elements(lambda x: split_mutant(x)[1], return_dtype=str).alias('mutant_part2')
])
azt_global_fitness_df = azt_global_fitness_df.with_columns([
    pl.col('mutant_profile').map_elements(lambda x: split_mutant(x)[0], return_dtype=str).alias('mutant_part1'),
    pl.col('mutant_profile').map_elements(lambda x: split_mutant(x)[1], return_dtype=str).alias('mutant_part2')
])


In [ ]:
amp_global_fitness_df = amp_global_fitness_df.sort("mutant_profile")
azt_global_fitness_df = azt_global_fitness_df.sort("mutant_profile")

In [ ]:
amp_global_fitness_df

In [ ]:
from repro_helpers import load_global_epistasis

# Per-drug global epistasis tables, sliced from data/processed/Epistasis_Combined.parquet
# at the representative concentration (AMP 781 / AZT 36), matching 05_epistasis_figures.ipynb.
epistasis_df, amp_global_epistasis_df, azt_global_epistasis_df = load_global_epistasis()

## Plot1: compare mutants' fitness in pairwise conditions


In [ ]:
def plot_pairwise_condition_comparison(
    stats_df: pl.DataFrame,
    nrows: int = 8,
    ncols: int = 6,
    width_per_col: float = 4,
    height_per_row: float = 3,
    bins: int = 64,
    plot_range: list = [[-5, 5], [-5, 5]],
    cmap: str = 'Blues'
) -> plt.Figure:
    """
    Create a grid of 2D histograms comparing mutant fitness between two drugs.
    
    Args:
        stats_df: Polars DataFrame containing drug comparison data
        nrows: Number of rows in the grid
        ncols: Number of columns in the grid
        width_per_col: Width in inches per column
        height_per_row: Height in inches per row
        bins: Number of bins for 2D histogram
        plot_range: Range for x and y axes as [[xmin, xmax], [ymin, ymax]]
        cmap: Colormap for 2D histogram
    
    Returns:
        matplotlib.figure.Figure: The created figure
    """
    # Calculate figure size
    fig_width = width_per_col * ncols
    fig_height = height_per_row * nrows
    
    # Create figure and axes grid
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, 
                            sharex=True, sharey=True, 
                            figsize=(fig_width, fig_height))
    
    # Get and sort drug concentrations
    amp_concs = stats_df.filter(pl.col("drug") == "Ampicillin")["concentration"].unique().to_numpy()
    azt_concs = stats_df.filter(pl.col("drug") == "Aztreonam")["concentration"].unique().to_numpy()
    
    amp_concs = sorted(amp_concs)
    azt_concs = sorted(azt_concs, reverse=True)
    
    # Create histograms
    for i, amp_c in enumerate(amp_concs):
        for j, azt_c in enumerate(azt_concs):
            # Get data for current concentrations
            x = stats_df.filter(pl.col("drug") == "Ampicillin")\
                       .filter(pl.col("concentration") == amp_c)["median_w"].to_numpy()
            y = stats_df.filter(pl.col("drug") == "Aztreonam")\
                       .filter(pl.col("concentration") == azt_c)["median_w"].to_numpy()
            
            # Create 2D histogram
            axes[j, i].hist2d(x, y, bins=bins, cmap=cmap, 
                            norm=colors.LogNorm(), range=plot_range)
            
            # Add labels
            axes[j, i].set_xlabel(f"Ampicillin: {amp_c}")
            axes[j, i].set_ylabel(f"Aztreonam: {azt_c}")
            
            # Add diagonal line
            axes[j, i].plot(plot_range[0], plot_range[0], 
                          color='red', linestyle='--', linewidth=1)
            
            # Set aspect ratio
            axes[j, i].set_aspect('equal', adjustable='box')
    
    # Adjust layout
    plt.tight_layout()
    
    return fig


In [ ]:
stats_df1 = amp_stats_df.with_columns(pl.Series(name="drug", values=["Ampicillin"]*amp_stats_df.shape[0]))
stats_df2 = azt_stats_df.with_columns(pl.Series(name="drug", values=["Aztreonam"]*azt_stats_df.shape[0]))

all_stats_df = pl.concat([stats_df1, stats_df2])
all_stats_df = all_stats_df.select(["mutant_profile", "concentration", "median", "median_w", "drug"])

all_stats_df = all_stats_df.sort(["mutant_profile", "concentration"])

fig = plot_pairwise_condition_comparison(all_stats_df, cmap="viridis", bins=200)
# plt.savefig(f"{base_path}/data/outputs/figures/summary-stats/pairwise_condition_comparison.png", bbox_inches="tight", dpi=500, format="png")
plt.show()

The conclusion from this analysis is that the population fitness decrease when we increase the concentration of the drug.

## Plot2: histogram of fitness across conditions

In [ ]:
def plot_drug_fitness_histograms(
    stats_df: pl.DataFrame,
    wildtype: str,
    dead_mutant: str,
    color: str,
    nrows: int = 6,
    figsize: tuple = (3, 9),
    nbins: int = 200,
    x_range: tuple = (0, 6),
    fontsize: dict = {"xlabel": 12, "concentration": 9}
) -> plt.Figure:
    """
    Create histograms of fitness distributions for different drug concentrations.
    
    Args:
        stats_df: Polars DataFrame containing drug fitness data
        wildtype: String identifier for wildtype sequence
        dead_mutant: String identifier for dead mutant sequence
        color: Color for histogram bars
        nrows: Number of rows in the subplot grid
        figsize: Figure size as (width, height)
        nbins: Number of bins for histograms
        x_range: Range for x-axis as (min, max)
        fontsize: Dictionary of font sizes for different elements
    
    Returns:
        matplotlib.figure.Figure: The created figure
    """
    # Create figure and axes
    fig, axes = plt.subplots(nrows=nrows, ncols=1, figsize=figsize, sharex=True)
    
    # Get concentrations and sort them
    concentrations = stats_df["concentration"].unique().to_numpy()
    concentrations = sorted(concentrations)
    
    # Get max y scale across all concentrations
    max_count = 0
    for conc in concentrations:
        data = stats_df.filter(pl.col("concentration") == conc).to_pandas()
        hist = np.histogram(data["median"], bins=nbins, range=x_range)
        max_count = max(max_count, max(hist[0]))
    y_max = max_count * 1.1  # Add 10% padding
    
    # Create histograms for each concentration
    for i, conc in enumerate(concentrations):
        data = stats_df.filter(pl.col("concentration") == conc).to_pandas()
        
        # Plot histogram
        sns.histplot(data, x="median", bins=nbins, kde=False, ax=axes[i],
                    color=color, alpha=0.7, binrange=x_range)
        
        # Add concentration label
        axes[i].text(0.02, 0.95, f"{conc} μg/μL",
                    transform=axes[i].transAxes,
                    fontsize=fontsize["concentration"],
                    verticalalignment='top')
        
        # Show wildtype and dead mutant medians
        wildtype_median = stats_df.filter(
            pl.col("mutant_profile") == wildtype
        ).filter(pl.col("concentration") == conc)["median"].to_numpy()[0]
        
        dead_mutant_median = stats_df.filter(
            pl.col("mutant_profile") == dead_mutant
        ).filter(pl.col("concentration") == conc)["median"].to_numpy()[0]
        
        # Add reference lines
        axes[i].axvline(wildtype_median, color="red", linestyle="--",
                       linewidth=0.8, alpha=0.7)
        axes[i].axvline(dead_mutant_median, color="black", linestyle="--",
                       linewidth=0.8, alpha=0.7)
        
        # Set axis limits
        axes[i].set_ylim(0, y_max)
        axes[i].set_xlim(*x_range)
        
        # Remove y-axis ticks and labels
        axes[i].set_yticks([])
        axes[i].set_ylabel("")
    
    # Set x-label on bottom subplot
    axes[-1].set_xlabel("Fitness", fontsize=fontsize["xlabel"])
    
    # Adjust layout
    plt.tight_layout()
    
    return fig

In [ ]:
# filter out rows whose median is 0.176091
amp_stats_df_filtered = amp_stats_df.filter(pl.col("median") > 0.1761)
azt_stats_df_filtered = azt_stats_df.filter(pl.col("median") > 0.1761)

fig = plot_drug_fitness_histograms(amp_stats_df_filtered, wildtype, dead_mutant, color="#3498DB", nrows=6)
plt.show()

In [ ]:
fig = plot_drug_fitness_histograms(azt_stats_df_filtered, wildtype, dead_mutant, color="#6B8E23", nrows=8)
plt.show()

In [ ]:
def plot_1concentration_high_fitness_histogram(
    stats_df: pl.DataFrame,
    concentration: float,
    fitness_threshold: float,
    figsize: tuple = (6, 3),
    color: str = None,
    bins: int = 200,
    fontsize: dict = {"xlabel": 12, "ylabel": 12}
) -> plt.Figure:
    """
    Create a histogram of fitness distribution for a specific drug concentration,
    showing only mutants above a certain fitness threshold.
    
    Args:
        stats_df: Polars DataFrame containing drug fitness data
        concentration: Drug concentration to plot
        fitness_threshold: Minimum fitness value to include
        figsize: Figure size as (width, height)
        color: Color for histogram bars
        bins: Number of bins for histogram
        fontsize: Dictionary of font sizes for different elements
    
    Returns:
        matplotlib.figure.Figure: The created figure
    """
    # Filter data for specific concentration
    conc_df = stats_df.filter(pl.col("concentration") == concentration)
    
    # Select mutants with high fitness
    high_fitness_df = conc_df.filter(pl.col("median") > fitness_threshold)
    
    # Create figure
    fig = plt.figure(figsize=figsize)
    
    # Create histogram
    sns.histplot(
        high_fitness_df.to_pandas(),
        x="median",
        bins=bins,
        kde=False,
        color=color
    )
    
    # Set labels
    plt.xlabel("Fitness", fontsize=fontsize["xlabel"])
    plt.ylabel("Count", fontsize=fontsize["ylabel"])
    
    # Optional: Add title (commented out as in original)
    # plt.title(f"Drug Concentration: {concentration}", fontsize=12, loc="left", pad=1)
    
    # Adjust layout
    plt.tight_layout()
    
    return fig

In [ ]:
fig = plot_1concentration_high_fitness_histogram(stats_df=azt_stats_df_filtered, concentration=108.0, fitness_threshold=3, color="#6B8E23")
plt.show()

In [ ]:
fig = plot_1concentration_high_fitness_histogram(stats_df=azt_stats_df_filtered, concentration=324.0, fitness_threshold=3, color="#6B8E23")
plt.show()

## Plot3: joyplot of fitness across concentrations

In [ ]:
pink_color = "#FF69B4"

In [ ]:
concentration = 12
fig = plot_1concentration_high_fitness_histogram(stats_df=azt_stats_df, concentration=concentration, fitness_threshold=3, color=pink_color)
plt.savefig(f"{base_path}/data/outputs/figures/summary-stats/high_fitness_azt_{concentration}.png", bbox_inches="tight", dpi=1000, format="png")
plt.show()  

In [ ]:
concentration = 36
fig = plot_1concentration_high_fitness_histogram(stats_df=azt_stats_df, concentration=concentration, fitness_threshold=3, color=pink_color)
plt.savefig(f"{base_path}/data/outputs/figures/summary-stats/high_fitness_azt_{concentration}.png", bbox_inches="tight", dpi=1000, format="png")
plt.show()  

In [ ]:
concentration = 108
fig = plot_1concentration_high_fitness_histogram(stats_df=azt_stats_df, concentration=concentration, fitness_threshold=3, color=pink_color)
plt.savefig(f"{base_path}/data/outputs/figures/summary-stats/high_fitness_azt_{concentration}.png", bbox_inches="tight", dpi=1000, format="png")
plt.show()  

In [ ]:
concentration = 324
fig = plot_1concentration_high_fitness_histogram(stats_df=azt_stats_df, concentration=concentration, fitness_threshold=3, color=pink_color)
plt.savefig(f"{base_path}/data/outputs/figures/summary-stats/high_fitness_azt_{concentration}.png", bbox_inches="tight", dpi=1000, format="png")
plt.show()  

In [ ]:
def plot_fitness_joyplot(stats_df, color, figsize=(6, 8), line_length=0.8):
    """Create a joyplot of fitness distributions across concentrations."""
    # Set font with fallback system fonts
    plt.rcParams.update({
        'font.family': 'sans-serif',
        'font.sans-serif': ['Arial', 'DejaVu Sans', 'Liberation Sans', 'Helvetica'],
        'font.size': 10,
        'axes.titlesize': 10,
        'axes.labelsize': 10,
        'xtick.labelsize': 9,
        'ytick.labelsize': 9
    })

    # Convert to pandas and prepare data with proper ordering
    plot_data = (stats_df
                 .select(['concentration', 'median', 'mutant_profile'])
                 .to_pandas())
    
    # Get concentrations in descending order and create categorical type
    concentrations = sorted(stats_df['concentration'].unique().to_numpy(), reverse=True)
    plot_data['concentration'] = pd.Categorical(
        plot_data['concentration'],
        categories=concentrations,
        ordered=True
    )
    
    # Sort data by concentration (descending)
    plot_data = plot_data.sort_values('concentration', ascending=False)
    
    # Calculate figure height based on number of concentrations
    height_per_subplot = 0.5
    total_height = height_per_subplot * len(concentrations) + 1
    
    # Create figure with calculated height
    plt.figure(figsize=(3, total_height))
    
    # Generate joyplot with sorted concentrations
    fig, axes = joypy.joyplot(
        data=plot_data,
        by="concentration",
        column="median",
        colormap=lambda x: color,
        labels=[None] * len(concentrations),  # Set labels to None initially
        range_style='all',
        tails=0.2,
        overlap=0,
        grid=False,
        fade=False,
        linecolor="k",
        linewidth=1,
        fill=True,
        figsize=(3, total_height),
        x_range=[0, 6]  # Set x range explicitly
    )
    
    # Update axes and add concentration labels on the right
    for idx, (ax, conc) in enumerate(zip(axes[:-1], concentrations)):  # Skip the last axis
        # Remove default y-axis labels and ticks
        ax.set_yticks([])
        
        # Get the y-limits of the current subplot
        ymin, ymax = ax.get_ylim()
        
        # Add concentration label on the right top corner
        # Position it at 90% of the height of the density curve
        ax.text(5.8, ymax * 0.9, f"{conc:.1f} μg/μL", 
                horizontalalignment='right',
                verticalalignment='top',
                transform=ax.transData,
                fontsize=10,
                fontfamily='sans-serif')
    
    # Set x-axis label with explicit font specification
    plt.xlabel("Fitness", fontsize=10, family='sans-serif')
    
    # Set x-axis ticks to show integers 1-5
    plt.xticks([0, 1, 2, 3, 4, 5, 6])
    
    # Add wildtype and dead mutant lines
    wildtype = "............."
    dead_mutant = "XXXXXXXXXXXXX"
    
    # Loop through concentrations and corresponding axes
    for idx, (conc, ax) in enumerate(zip(concentrations, axes[:-1])):
        # Get reference values
        wildtype_median = (stats_df
                           .filter(pl.col("mutant_profile") == wildtype)
                           .filter(pl.col("concentration") == conc)
                           ["median"].to_numpy()[0])
        dead_mutant_median = (stats_df
                              .filter(pl.col("mutant_profile") == dead_mutant)
                              .filter(pl.col("concentration") == conc)
                              ["median"].to_numpy()[0])
        
        # Calculate y coordinates based on overlap
        y_bottom = 0
        y_top = line_length

        print(wildtype_median, dead_mutant_median)

        # Add vertical lines to specific axis with correct y range
        ax.plot([wildtype_median, wildtype_median], [y_bottom, y_top],
                color="black", linestyle="--", linewidth=2, alpha=1, zorder=10)
        ax.plot([dead_mutant_median, dead_mutant_median], [y_bottom, y_top],
                color="blue", linestyle="--", linewidth=2, alpha=1, zorder=10)

    return fig

In [ ]:
# For ampicillin plots
grey_color = "#808080"
fig = plot_fitness_joyplot(amp_long_df, color=grey_color, line_length=3.5)
plt.savefig(f"{base_path}/data/outputs/figures/summary-stats/joyplot_amp.png", bbox_inches="tight", dpi=1000, format="png")
plt.show()

In [ ]:
# For aztreonam plots  
pink_color = "#FF69B4"
fig = plot_fitness_joyplot(azt_long_df, color=pink_color, line_length=3.5) 
plt.savefig(f"{base_path}/data/outputs/figures/summary-stats/joyplot_azt.png", bbox_inches="tight", dpi=1000, format="png")
plt.show()

In [ ]:
def plot_high_fitness_joyplot(stats_df, color, figsize=(6, 8), line_length=0.8):
    """Create a joyplot of fitness distributions focused on high fitness values (>3.5)."""
    # Set font with fallback system fonts
    plt.rcParams.update({
        'font.family': 'sans-serif',
        'font.sans-serif': ['Arial', 'DejaVu Sans', 'Liberation Sans', 'Helvetica'],
        'font.size': 10,
        'axes.titlesize': 10,
        'axes.labelsize': 10,
        'xtick.labelsize': 9,
        'ytick.labelsize': 9
    })

    # Convert to pandas and prepare data with proper ordering
    plot_data = (stats_df
                 .select(['concentration', 'median', 'mutant_profile'])
                 .to_pandas())
    
    # Get concentrations in descending order and create categorical type
    concentrations = sorted(stats_df['concentration'].unique().to_numpy(), reverse=True)
    plot_data['concentration'] = pd.Categorical(
        plot_data['concentration'],
        categories=concentrations,
        ordered=True
    )
    
    # Sort data by concentration (descending)
    plot_data = plot_data.sort_values('concentration', ascending=False)
    
    # Calculate figure height based on number of concentrations
    height_per_subplot = 0.5
    total_height = height_per_subplot * len(concentrations) + 1
    
    # Create figure with calculated height
    plt.figure(figsize=(3, total_height))
    
    # Generate joyplot with sorted concentrations
    fig, axes = joypy.joyplot(
        data=plot_data,
        by="concentration",
        column="median",
        colormap=lambda x: color,
        labels=[None] * len(concentrations),  # Set labels to None initially
        range_style='all',
        tails=0.2,
        overlap=0,
        grid=False,
        fade=False,
        linecolor="k",
        linewidth=1,
        fill=True,
        figsize=(3, total_height),
        x_range=[3.5, 6]  # Focus on high fitness values (>3.5)
    )
    
    # Update axes and add concentration labels on the right
    for idx, (ax, conc) in enumerate(zip(axes[:-1], concentrations)):  # Skip the last axis
        # Remove default y-axis labels and ticks
        ax.set_yticks([])
        
        # Get the y-limits of the current subplot
        ymin, ymax = ax.get_ylim()
        
        # Add concentration label on the right top corner
        # Position it at 90% of the height of the density curve
        ax.text(5.8, ymax * 0.9, f"{conc:.1f} μg/μL", 
                horizontalalignment='right',
                verticalalignment='top',
                transform=ax.transData,
                fontsize=10,
                fontfamily='sans-serif')
    
    # Set x-axis label with explicit font specification
    plt.xlabel("Fitness", fontsize=10, family='sans-serif')
    
    # Set x-axis ticks to show range from 3.5 to 6
    plt.xticks([3.5, 4, 4.5, 5, 5.5, 6])
    
    # Add wildtype and dead mutant lines
    wildtype = "............."
    dead_mutant = "XXXXXXXXXXXXX"
    
    # Loop through concentrations and corresponding axes
    for idx, (conc, ax) in enumerate(zip(concentrations, axes[:-1])):
        # Get reference values
        wildtype_median = (stats_df
                           .filter(pl.col("mutant_profile") == wildtype)
                           .filter(pl.col("concentration") == conc)
                           ["median"].to_numpy()[0])
        dead_mutant_median = (stats_df
                              .filter(pl.col("mutant_profile") == dead_mutant)
                              .filter(pl.col("concentration") == conc)
                              ["median"].to_numpy()[0])
        
        # Calculate y coordinates based on overlap
        y_bottom = 0
        y_top = line_length

        print(wildtype_median, dead_mutant_median)

        # Add vertical lines to specific axis with correct y range
        ax.plot([wildtype_median, wildtype_median], [y_bottom, y_top],
                color="black", linestyle="--", linewidth=2, alpha=1, zorder=10)
        ax.plot([dead_mutant_median, dead_mutant_median], [y_bottom, y_top],
                color="blue", linestyle="--", linewidth=2, alpha=1, zorder=10)

    return fig

In [ ]:
# For ampicillin plots
grey_color = "#808080"
fig = plot_high_fitness_joyplot(amp_long_df, color=grey_color, line_length=3.5)
plt.show()

In [ ]:
# For aztreonam plots  
pink_color = "#FF69B4"
fig = plot_high_fitness_joyplot(azt_long_df, color=pink_color, line_length=3.5) 
plt.show()

## Density plot for global fitness

In [ ]:
def plot_fitness_distribution(
    df: pl.DataFrame,
    drug_name: str,
    color: str,
    figsize: tuple = (6, 4),
    hist_alpha: float = 0.7,
    bins: int = 30,
    show_grid: bool = False,
    save_path: str = None
) -> plt.Figure:
    """
    Create a histogram plot for global fitness distribution.
    
    Parameters:
    -----------
    df : polars.DataFrame
        DataFrame containing the fitness data with normalized_fitness column
    drug_name : str
        Name of the drug for plot title
    color : str
        Color for histogram
    figsize : tuple
        Figure size in inches (width, height)
    hist_alpha : float
        Transparency for histogram bars
    bins : int
        Number of bins for histogram
    show_grid : bool
        Whether to show grid lines
    save_path : str, optional
        If provided, save figure to this path
        
    Returns:
    --------
    matplotlib.figure.Figure
        The created figure
    """
    # Set font with fallback system fonts
    plt.rcParams.update({
        'font.family': 'sans-serif',
        'font.sans-serif': ['Arial', 'DejaVu Sans', 'Liberation Sans', 'Helvetica'],
        'font.size': 10,
        'axes.titlesize': 10,
        'axes.labelsize': 10,
        'xtick.labelsize': 9,
        'ytick.labelsize': 9
    })
    
    # Create figure
    fig, ax = plt.subplots(figsize=figsize)
    
    # Convert to pandas for seaborn
    data = df.to_pandas()
    
    # Plot histogram
    sns.histplot(
        data=data,
        x='normalized_fitness',
        bins=bins,
        color=color,
        alpha=hist_alpha,
        stat='count',  # Show counts instead of density
        ax=ax
    )
    
    # Get wildtype and dead mutant global fitness values
    wildtype = "............."
    dead_mutant = "XXXXXXXXXXXXX"
    
    wildtype_fitness = df.filter(pl.col("mutant_profile") == wildtype)["normalized_fitness"].item()
    dead_mutant_fitness = df.filter(pl.col("mutant_profile") == dead_mutant)["normalized_fitness"].item()
    
    # Add vertical lines for wildtype and dead mutant
    ymin, ymax = ax.get_ylim()
    ax.vlines(x=wildtype_fitness, ymin=0, ymax=ymax*0.8, 
              color='black', linestyle='--', linewidth=1, alpha=1)
    ax.vlines(x=dead_mutant_fitness, ymin=0, ymax=ymax*0.8, 
              color='blue', linestyle='--', linewidth=1, alpha=1)
    
    # Set labels and title
    ax.set_title(f'{drug_name} Global Fitness Distribution')
    ax.set_xlabel('$Fitness_{global}$')  # LaTeX style
    ax.set_ylabel('Count')
    
    # Add grid if requested
    if show_grid:
        ax.grid(True, alpha=0.3)
    
    # Adjust layout
    plt.tight_layout()
    
    # Save figure if path provided
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    
    return fig

In [ ]:
grey_color = "#808080"
fig_amp = plot_fitness_distribution(
    df=amp_global_fitness_df,
    drug_name='Ampicillin',
    color=grey_color,
    figsize=(3.5, 2.5),
    bins=50,
    show_grid=False
)
# fig_amp.savefig(f"{base_path}/data/outputs/figures/summary-stats/histogram_global_fitness_amp.png", dpi=1000, bbox_inches='tight', format="png")
plt.show()

In [ ]:
# Example usage for Aztreonam
pink_color = "#FF69B4"
fig_azt = plot_fitness_distribution(
    df=azt_global_fitness_df,
    drug_name='Aztreonam',
    color=pink_color,
    figsize=(3.5, 2.5),
    bins=50,
    show_grid=False
)
# fig_azt.savefig(f"{base_path}/data/outputs/figures/summary-stats/histogram_global_fitness_azt.png", dpi=1000, bbox_inches='tight', format="png")
plt.show()

In [ ]:
def plot_global_high_fitness_histogram(
    stats_df: pl.DataFrame,
    fitness_threshold: float,
    figsize: tuple = (6, 3),
    color: str = None,
    bins: int = 200,
    fontsize: dict = {"xlabel": 12, "ylabel": 12}
) -> plt.Figure:
    """
    Create a histogram of global fitness distribution,
    showing only mutants above a certain fitness threshold.
    
    Args:
        stats_df: Polars DataFrame containing global fitness data
        fitness_threshold: Minimum fitness value to include
        figsize: Figure size as (width, height)
        color: Color for histogram bars
        bins: Number of bins for histogram
        fontsize: Dictionary of font sizes for different elements
    
    Returns:
        matplotlib.figure.Figure: The created figure
    """
    # Select mutants with high fitness
    high_fitness_df = stats_df.filter(pl.col("normalized_fitness") > fitness_threshold)
    
    # Create figure
    fig = plt.figure(figsize=figsize)
    
    # Create histogram
    sns.histplot(
        high_fitness_df.to_pandas(),
        x="normalized_fitness",
        bins=bins,
        kde=False,
        color=color
    )
    
    # Set labels
    plt.xlabel("$Fitness_{global}$", fontsize=fontsize["xlabel"])
    plt.ylabel("Count", fontsize=fontsize["ylabel"])
    
    # Adjust layout
    plt.tight_layout()
    
    return fig

In [ ]:
pink_color = "#FF69B4"
fig = plot_global_high_fitness_histogram(
    stats_df=azt_global_fitness_df,
    fitness_threshold=2.0,
    color=pink_color,
    figsize=(6, 3),
    bins=50
)
plt.savefig(f"{base_path}/data/outputs/figures/summary-stats/histogram_part_global_fitness_azt.png", dpi=1000, bbox_inches='tight', format="png")
plt.show()

## Plot4: fitness with or without a mutation in each condition

In [ ]:
intended = {19: ['.','P'], 37: ['.','K'], 67: ['.','L','V'],
            102: ['.','K'], 162: ['.','S','H','N'],
            180: ['.','T'], 235: ['.','T'], 236: ['.','S'],
            237: ['.','K'], 241: ['.','S','C'], 261: ['.','M'],
            271: ['.','L','Q'], 272:['.','D']}

# convert the intended dict to a list of tuples
mutations = []
for item in intended.items():
    pos = item[0]
    aas = item[1]
    for aa in aas:
        if aa != ".":
            mutations.append((pos, aa))

mutations

In [ ]:
def get_mutation_pairs(mutant_profiles, mutation_pos, mutation_aa):
    all_mutations = {19: ['.','P'], 37: ['.','K'], 67: ['.','L','V'],
            102: ['.','K'], 162: ['.','S','H','N'],
            180: ['.','T'], 235: ['.','T'], 236: ['.','S'],
            237: ['.','K'], 241: ['.','S','C'], 261: ['.','M'],
            271: ['.','L','Q'], 272:['.','D']}
    
    pos_list = list(all_mutations.keys())
    # get the id of pos in the list
    mutation_index = pos_list.index(mutation_pos)
    
    pairs = []
    profile_dict = {}
    
    for profile in mutant_profiles:
        # Create a key that represents the profile without the mutation position
        key = profile[:mutation_index] + profile[mutation_index+1:]
        
        if profile[mutation_index] == mutation_aa:
            if key in profile_dict:
                pairs.append((profile, profile_dict[key]))
            else:
                profile_dict[key] = profile
        elif profile[mutation_index] == '.':
            if key in profile_dict:
                pairs.append((profile_dict[key], profile))
            else:
                profile_dict[key] = profile
    
    # Assertions to check pairs
    for with_mutation, without_mutation in pairs:
        assert with_mutation[mutation_index] == mutation_aa, f"Expected {mutation_aa} at position {mutation_pos} in {with_mutation}"
        assert without_mutation[mutation_index] == '.', f"Expected '.' at position {mutation_pos} in {without_mutation}"
        assert with_mutation[:mutation_index] == without_mutation[:mutation_index], f"Mismatch before mutation position in {with_mutation} and {without_mutation}"
        assert with_mutation[mutation_index+1:] == without_mutation[mutation_index+1:], f"Mismatch after mutation position in {with_mutation} and {without_mutation}"
        assert sum(c1 != c2 for c1, c2 in zip(with_mutation, without_mutation)) == 1, f"More than one difference between {with_mutation} and {without_mutation}"

    return pairs

In [ ]:
def plot_mutation_fitness_comparison(
    stats_df: pl.DataFrame,
    drug_name: str,
    figsize_per_plot: tuple = (4, 3),
    bins: int = 64,
    plot_range: list = [[-5, 5], [-5, 5]],
    cmap: str = 'Blues',
    intended_mutations: dict = None
) -> plt.Figure:
    """
    Create a grid of 2D histograms comparing fitness with/without specific mutations
    across different drug concentrations.
    
    Args:
        stats_df: Polars DataFrame containing fitness data
        drug_name: Name of the drug (e.g., "Amp" or "Azt")
        figsize_per_plot: Size of each subplot as (width, height)
        bins: Number of bins for 2D histograms
        plot_range: Range for x and y axes as [[xmin, xmax], [ymin, ymax]]
        cmap: Colormap for 2D histograms
        intended_mutations: Dictionary of intended mutations (if None, uses default)
    
    Returns:
        matplotlib.figure.Figure: The created figure
    """
    
    # Create list of mutations
    mutations = [(pos, aa) for pos, aas in intended_mutations.items() 
                for aa in aas if aa != '.']
    
    # Sort concentrations
    concentrations = sorted(stats_df["concentration"].unique().to_numpy())
    
    # Pre-process the data
    median_dict = stats_df.select(["mutant_profile", "concentration", "median_w"])\
                         .to_dict(as_series=False)
    median_dict = {(profile, conc): median 
                  for profile, conc, median in zip(median_dict["mutant_profile"], 
                                                 median_dict["concentration"], 
                                                 median_dict["median_w"])}
    
    # Calculate grid dimensions
    n_rows = len(mutations)
    n_cols = len(concentrations)
    
    # Create figure and axes grid
    fig, axes = plt.subplots(nrows=n_rows, 
                            ncols=n_cols,
                            figsize=(figsize_per_plot[0]*n_cols, 
                                   figsize_per_plot[1]*n_rows),
                            sharex=True, 
                            sharey=True)
    
    # Create plots for each mutation and concentration
    for i, (mutation_pos, mutation_aa) in enumerate(mutations):
        pairs = get_mutation_pairs(stats_df["mutant_profile"].unique().to_list(),
                                 mutation_pos, 
                                 mutation_aa)
        
        for j, conc in enumerate(concentrations):
            medians_with_mutation = [median_dict.get((with_mut, conc), np.nan) 
                                   for with_mut, _ in pairs]
            medians_without_mutation = [median_dict.get((without_mut, conc), np.nan) 
                                      for _, without_mut in pairs]
            
            # Create 2D histogram
            axes[i, j].hist2d(medians_with_mutation, 
                            medians_without_mutation,
                            bins=bins,
                            cmap=cmap,
                            norm=colors.LogNorm(),
                            range=plot_range)
            
            # Add labels
            if i == n_rows - 1:
                axes[i, j].set_xlabel(f"{drug_name}: {conc}")
            if j == 0:
                axes[i, j].set_ylabel(f"{mutation_pos}{mutation_aa}")
            
            # Add diagonal line
            axes[i, j].plot(plot_range[0], plot_range[0],
                          'r--',
                          alpha=0.75,
                          zorder=0)
    
    # Adjust layout
    plt.tight_layout()
    
    return fig

In [ ]:
fig = plot_mutation_fitness_comparison(
    stats_df=azt_stats_df_filtered,
    drug_name="Aztreonam",
    intended_mutations=intended,
    figsize_per_plot=(4, 3),
    bins=200,
    plot_range=[[-5, 5], [-5, 5]],
    cmap="viridis"
)
plt.savefig(f"{base_path}/data/outputs/figures/summary-stats/mutation_fitness_comparison_azt.png", bbox_inches="tight", dpi=500, format="png")
plt.show()

In [ ]:
fig = plot_mutation_fitness_comparison(
    stats_df=amp_stats_df_filtered,
    drug_name="Ampicillin",
    intended_mutations=intended,
    figsize_per_plot=(4, 3),
    bins=200,
    plot_range=[[-5, 5], [-5, 5]],
    cmap="viridis"
)
plt.savefig(f"{base_path}/data/outputs/figures/summary-stats/mutation_fitness_comparison_amp.png", bbox_inches="tight", dpi=500, format="png")
plt.show()

## Plot5: Show data scale using heatmap

In [ ]:
def plot_fitness_heatmap(long_df, figsize=(10, 6), vmin=None, vmax=None, show_n_labels=10, cmap="RdBu_r"):
    """
    Create a heatmap of fitness values across concentrations and genotypes.
    Values are normalized per concentration level.
    Genotypes are sorted by number of mutations and labeled with a color bar.
    """
    # Convert to pandas and pivot to wide format
    pivot_df = (long_df
                .select(['concentration', 'mutant_profile', 'median'])
                .to_pandas()
                .pivot(index='concentration', 
                       columns='mutant_profile', 
                       values='median'))
    
    # Sort columns by number of mutations
    def count_mutations(genotype):
        return sum(1 for c in genotype if c != '.')
    
    # Sort columns and store mutation counts
    mutation_counts = [(col, count_mutations(col)) for col in pivot_df.columns]
    sorted_columns = sorted(mutation_counts, key=lambda x: (x[1], x[0]))
    pivot_df = pivot_df.reindex(columns=[col for col, _ in sorted_columns])
    # This will now display genotypes ordered by mutation count from left to right: 
    # wildtype first, then single mutations, then double mutations, etc. 
    # Within each mutation count group, genotypes are sorted alphabetically for consistency.

    # Create figure with two subplots (one for mutation counts, one for heatmap)
    fig = plt.figure(figsize=figsize)
    gs = plt.GridSpec(2, 1, height_ratios=[1, 10], hspace=0.05)
    
    # Main subplot for heatmap
    ax_main = fig.add_subplot(gs[1])
    
    # Create heatmap in main subplot
    heatmap = sns.heatmap(pivot_df,
                         cmap=cmap,
                         center=2.5,
                         cbar_kws={'label': 'Normalized Fitness'},
                         xticklabels=False,
                         ax=ax_main)
    
    # Get the position of the heatmap
    heatmap_pos = ax_main.get_position()
    
    # Top subplot for mutation count visualization
    ax_top = fig.add_subplot(gs[0])
    
    # Set the position of top subplot to align with heatmap
    ax_top.set_position([heatmap_pos.x0, ax_top.get_position().y0,
                        heatmap_pos.width, ax_top.get_position().height])
    
    # Create array of mutation counts for visualization
    counts = [count for _, count in sorted_columns]
    unique_counts = sorted(set(counts))
    count_colors = plt.cm.tab20(np.linspace(0, 1, len(unique_counts)))
    
    # Create color blocks for mutation counts
    left = 0
    for count in unique_counts:
        width = counts.count(count)
        ax_top.barh(y=0, width=width, left=left, height=1, 
                   color=count_colors[unique_counts.index(count)],
                   align='edge')
        
        # Add count label in the middle of each block
        if width > len(pivot_df.columns) * 0.05:  # Only label if block is wide enough
            ax_top.text(left + width/2, 0.5, str(count),
                       ha='center', va='center')
        left += width
    
    # Style the top subplot
    ax_top.set_xlim(0, len(counts))
    ax_top.set_ylim(0, 1)
    ax_top.set_xticks([])
    ax_top.set_yticks([])
    ax_top.set_title('Number of mutations', pad=2)

    # # Show only subset of x-labels
    # n_cols = len(pivot_df.columns)
    # step = max(n_cols // show_n_labels, 1)
    # plt.xticks(range(0, n_cols, step), 
    #            [pivot_df.columns[i] for i in range(0, n_cols, step)],
    #            rotation=45, 
    #            ha='right',
    #            fontfamily='monospace',
    #            fontsize=8)

    # Style the main subplot
    ax_main.set_xlabel('Genotypes')
    ax_main.set_ylabel('Concentration')
    ax_main.set_yticks(ax_main.get_yticks())
    ax_main.set_yticklabels(ax_main.get_yticklabels(), rotation=0)
    
    return fig

In [ ]:
plot_fitness_heatmap(amp_long_df, 
                    figsize=(15, 3),
                    vmin=0, 
                    vmax=6,
                    show_n_labels=50,
                    cmap="PiYG")
# plt.savefig(f"{base_path}/data/outputs/figures/summary-stats/heatmap_amp.png", dpi=1000, bbox_inches='tight', format="png")
plt.show()

In [ ]:
plot_fitness_heatmap(azt_long_df, 
                    figsize=(15, 3),
                    vmin=0, 
                    vmax=6,
                    show_n_labels=50,
                    cmap="PiYG")
# plt.savefig(f"{base_path}/data/outputs/figures/summary-stats/heatmap_azt.png", dpi=1000, bbox_inches='tight', format="png")
plt.show()

## Plot6: show fitness pattern for amino acid states using heat map

In [ ]:
amp_global_fitness_df

In [ ]:
# Sort azt_global_fitness_df by normalized_fitness in descending order
azt_global_fitness_df.sort("normalized_fitness", descending=True)


In [ ]:
# dead_mutant = "XXXXXXXXXXXXX"
# heatmap_df = amp_global_fitness_df.filter(pl.col("mutant_profile") != dead_mutant)

# fig = px.density_heatmap(heatmap_df.to_pandas(), x="mutant_part2", y="mutant_part1", z="normalized_fitness", 
#                         color_continuous_scale="RdBu_r", range_color=[0, 2])

# fig.update_layout(
#     width=1100,
#     height=800,
# )

# fig.update_xaxes(tickfont=dict(family="Courier New, monospace"))
# fig.update_yaxes(tickfont=dict(family="Courier New, monospace"))

# # fig.show()

In [ ]:

# heatmap_df = azt_global_fitness_df.filter(pl.col("mutant_profile") != dead_mutant)

# fig = px.density_heatmap(heatmap_df.to_pandas(), x="mutant_part2", y="mutant_part1", z="normalized_fitness", 
#                         color_continuous_scale="RdBu_r", range_color=[-4, 6])

# fig.update_layout(
#     width=1100,
#     height=800,
# )

# fig.update_xaxes(tickfont=dict(family="Courier New, monospace"))
# fig.update_yaxes(tickfont=dict(family="Courier New, monospace"))

# # fig.show()


In [ ]:
# # Code for animation

# fig = px.density_heatmap(amp_stats_df.filter(pl.col("mutant_profile") != dead_mutant).to_pandas(), x="mutant_part2", y="mutant_part1", z="median_w",
#                          animation_frame="concentration", color_continuous_scale="RdBu_r", range_color=[-4, 4]) 

# fig.update_layout(
#     width=1000,
#     height=800,
# )

# fig.update_xaxes(tickfont=dict(family="Courier New, monospace"))
# fig.update_yaxes(tickfont=dict(family="Courier New, monospace"))
# fig.show()

In [ ]:
def plot_concentration_heatmaps(stats_df, height=4, vmin=0, vmax=6, cmap=None):
    """
    Create a row of square heatmaps for each concentration level with a single colorbar.
    
    Parameters:
    -----------
    stats_df : polars.DataFrame
        DataFrame containing concentration, mutant_part1, mutant_part2, and fitness data
    height : float
        Height of each heatmap (width will be the same for square aspect)
    vmin, vmax : float
        Min and max values for color scaling
    cmap : matplotlib colormap
        Custom colormap for heatmap (defaults to custom diverging palette)
    """
    # remove dead mutants
    stats_df = stats_df.filter(pl.col("mutant_profile") != "XXXXXXXXXXXXX")
    # Helper function to count mutations
    def count_mutations(genotype):
        return sum(1 for c in genotype if c != '.')
    
    # Get unique concentrations
    concentrations = sorted(stats_df['concentration'].unique())
    n_concentrations = len(concentrations)
    
    # Calculate figure width
    figsize = (height * n_concentrations * 1.3, height)
    
    # Create figure and subplots
    fig, axes = plt.subplots(1, n_concentrations, figsize=figsize, 
                            constrained_layout=True)
    
    # Store the last heatmap for colorbar
    last_heatmap = None
    
    # Create heatmap for each concentration
    for idx, concentration in enumerate(concentrations):
        # Filter data for current concentration
        conc_df = stats_df.filter(pl.col('concentration') == concentration)
        
        # Create pivot table
        pivot_df = conc_df.to_pandas().pivot_table(
            values='median',
            index='mutant_part2',
            columns='mutant_part1',
            aggfunc='mean'
        )
        
        # Sort both rows and columns by mutation count and alphabetically
        row_mutations = [(idx, count_mutations(idx)) for idx in pivot_df.index]
        col_mutations = [(col, count_mutations(col)) for col in pivot_df.columns]
        
        sorted_rows = sorted(row_mutations, key=lambda x: (x[1], x[0]))
        sorted_cols = sorted(col_mutations, key=lambda x: (x[1], x[0]))
        
        pivot_df = pivot_df.reindex(
            index=[row for row, _ in sorted_rows],
            columns=[col for col, _ in sorted_cols]
        )
        
        # Create heatmap without colorbar
        last_heatmap = sns.heatmap(pivot_df,
                                  cmap=cmap,
                                  center=2.5,
                                  vmin=vmin,
                                  vmax=vmax,
                                  ax=axes[idx],
                                  cbar=False)
        
        # Update labels and style
        axes[idx].set_title(f'Conc: {concentration}')
        axes[idx].set_xlabel('Mutant Part 1')
        axes[idx].set_ylabel('Mutant Part 2')
        
        # Update tick labels
        axes[idx].set_xticklabels(axes[idx].get_xticklabels(), 
                                 fontfamily='monospace', 
                                 rotation=45, 
                                 ha='right',
                                 fontsize=8)
        axes[idx].set_yticklabels(axes[idx].get_yticklabels(), 
                                 fontfamily='monospace', 
                                 rotation=0,
                                 fontsize=8)
    
    # Add a single colorbar for all subplots
    fig.colorbar(last_heatmap.collections[0], ax=axes, location='right', label='Fitness')
    
    return fig, axes

In [ ]:
# Create heatmaps for amp_stats_df
fig, axes = plot_concentration_heatmaps(amp_stats_df, 
                                      height=4,
                                      vmin=0, 
                                      vmax=6,
                                      cmap="PiYG")
# plt.savefig(f"{base_path}/data/outputs/figures/summary-stats/parts_heatmap_amp.png", dpi=1000, bbox_inches='tight', format="png")
plt.show()

In [ ]:
# Create heatmaps for amp_stats_df
fig, axes = plot_concentration_heatmaps(azt_stats_df, 
                                        height=4,
                                        vmin=0, 
                                        vmax=6,
                                        cmap="PiYG")
# plt.savefig(f"{base_path}/data/outputs/figures/summary-stats/parts_heatmap_azt.png", dpi=700, bbox_inches='tight', format="png")
plt.show()

In [ ]:
def plot_zoomed_concentration_heatmap(
    stats_df,
    concentration,
    max_mutations=2,
    height=8,
    vmin=0,
    vmax=6,
    cmap=None,
    fontsize=10
):
    """
    Create a zoomed-in heatmap for a specific concentration focusing on low mutation counts.
    
    Parameters:
    -----------
    stats_df : polars.DataFrame
        DataFrame containing concentration, mutant_part1, mutant_part2, and fitness data
    concentration : float
        Specific concentration level to plot
    max_mutations : int
        Maximum number of mutations to include in visualization
    height : float
        Height of the heatmap (width will adjust to maintain square cells)
    vmin, vmax : float
        Min and max values for color scaling
    cmap : matplotlib colormap
        Custom colormap for heatmap
    fontsize : int
        Font size for genotype labels
    """
    # Remove dead mutants and filter for specific concentration
    filtered_df = (stats_df
                  .filter(pl.col("mutant_profile") != "XXXXXXXXXXXXX")
                  .filter(pl.col("concentration") == concentration))
    
    # Create pivot table
    pivot_df = filtered_df.to_pandas().pivot_table(
        values='median',
        index='mutant_part2',
        columns='mutant_part1',
        aggfunc='mean'
    )
    
    # Helper function to count mutations
    def count_mutations(genotype):
        return sum(1 for c in genotype if c != '.')
    
    # Filter and sort rows/columns by mutation count
    row_mutations = [(idx, count_mutations(idx)) for idx in pivot_df.index]
    col_mutations = [(col, count_mutations(col)) for col in pivot_df.columns]
    
    # Filter for low mutation counts
    filtered_rows = [row for row, count in row_mutations if count <= max_mutations]
    filtered_cols = [col for col, count in col_mutations if count <= max_mutations]
    
    # Sort filtered rows/cols by mutation count then alphabetically
    sorted_rows = sorted(filtered_rows, key=lambda x: (count_mutations(x), x))
    sorted_cols = sorted(filtered_cols, key=lambda x: (count_mutations(x), x))
    
    # Reindex the dataframe
    pivot_df = pivot_df.reindex(index=sorted_rows, columns=sorted_cols)
    
    # Calculate figure size to maintain square cells
    n_rows, n_cols = len(sorted_rows), len(sorted_cols)
    width = height * (n_cols / n_rows) * 1.3  # 1.3 factor for colorbar space
    
    # Create figure
    fig, ax = plt.subplots(figsize=(width, height))
    
    # Create heatmap
    heatmap = sns.heatmap(pivot_df,
                         cmap=cmap,
                         center=2.5,
                         vmin=vmin,
                         vmax=vmax,
                         ax=ax,
                         cbar_kws={'label': 'Fitness'},
                         square=True)
    
    # Update labels and style
    ax.set_title(f'Concentration: {concentration}')
    ax.set_xlabel('Mutant Part 1')
    ax.set_ylabel('Mutant Part 2')
    
    # Update tick labels with increased font size and spacing
    ax.set_xticklabels(ax.get_xticklabels(),
                       fontfamily='monospace',
                       rotation=45,
                       ha='right',
                       fontsize=fontsize)
    ax.set_yticklabels(ax.get_yticklabels(),
                       fontfamily='monospace',
                       rotation=45,  # Changed from 0 to 45
                       ha='right',   # Added horizontal alignment
                       fontsize=fontsize)
    
    # Adjust layout to prevent label cutoff
    plt.subplots_adjust(left=0.2)  # Added more space on the left
    
    return fig, ax

In [ ]:
fig, ax = plot_zoomed_concentration_heatmap(
    stats_df=azt_stats_df,
    concentration=324.0,
    max_mutations=3,
    height=8.5,
    cmap="PiYG"
)
# plt.savefig(f"{base_path}/data/outputs/figures/summary-stats/zoomin_heatmap_amp_781.0.png", dpi=1000, bbox_inches='tight', format="png")
plt.show()

## Heatmap Global Fitness

In [ ]:
def plot_global_normalized_fitness_heatmap(
    stats_df,
    figsize=(10, 10),
    vmin=0,
    vmax=6,
    cmap="PiYG",
    fontsize=10
):
    """
    Create a heatmap for global normalized fitness with consistent sorting and color scale.
    
    Parameters:
    -----------
    stats_df : polars.DataFrame
        DataFrame containing global fitness data with columns:
        'normalized_fitness', 'mutant_part1', 'mutant_part2', and 'mutant_profile'
    figsize : tuple
        Overall figure dimensions (width, height)
    vmin, vmax : float
        Min and max values for color scale
    cmap : str
        Colormap to use
    fontsize : int
        Font size for labels
    """
    import matplotlib.pyplot as plt
    from matplotlib import gridspec, colors
    import seaborn as sns
    import numpy as np
    import polars as pl

    # Helper function to count mutations
    def count_mutations(genotype):
        return sum(1 for c in genotype if c != '.')
    
    # Pivot the data
    pivot_df = stats_df.to_pandas().pivot_table(
        values="normalized_fitness",
        index="mutant_part2",
        columns="mutant_part1",
        aggfunc="mean"
    )
    
    # Sort both rows and columns by mutation count and alphabetically
    row_mutations = [(idx, count_mutations(idx)) for idx in pivot_df.index]
    col_mutations = [(col, count_mutations(col)) for col in pivot_df.columns]
    
    sorted_rows = sorted(row_mutations, key=lambda x: (x[1], x[0]), reverse=True)  # Reverse order for rows
    sorted_cols = sorted(col_mutations, key=lambda x: (x[1], x[0]))  # Keep same order for columns
    
    pivot_df = pivot_df.reindex(
        index=[row for row, _ in sorted_rows],
        columns=[col for col, _ in sorted_cols]
    )
    
    # Get wildtype and dead mutant values
    wildtype = "............."
    dead_mutant = "XXXXXXXXXXXXX"
    
    wildtype_value = stats_df.filter(pl.col("mutant_profile") == wildtype)["normalized_fitness"].item()
    dead_value = stats_df.filter(pl.col("mutant_profile") == dead_mutant)["normalized_fitness"].item()
    
    # Create figure with GridSpec
    fig = plt.figure(figsize=figsize)
    gs = gridspec.GridSpec(nrows=1, ncols=2, width_ratios=[1, 0.05], wspace=0.05)
    ax = fig.add_subplot(gs[0])
    cax = fig.add_subplot(gs[1])
    
    # Create the heatmap
    norm = colors.TwoSlopeNorm(vmin=vmin, vcenter=wildtype_value, vmax=vmax)
    
    heatmap = sns.heatmap(
        pivot_df,
        cmap=cmap,
        norm=norm,
        cbar=True,
        ax=ax,
        cbar_ax=cax,
        cbar_kws={"label": "Global Normalized Fitness"}
    )
    
    # Modify the color bar ticks and labels
    cbar = heatmap.collections[0].colorbar
    cbar.set_ticks([vmin, wildtype_value, vmax])
    cbar.set_ticklabels([
        f"{vmin:.1f}\n(Dead Mutant)",
        f"{wildtype_value:.1f}\n(Wildtype)",
        f"{vmax:.1f}"
    ])
    
    # Set labels
    ax.set_title("Global Normalized Fitness", fontsize=fontsize)
    ax.set_xlabel("Mutant Part 1", fontsize=fontsize)
    ax.set_ylabel("Mutant Part 2", fontsize=fontsize)
    
    # Format tick labels
    ax.set_xticklabels(
        ax.get_xticklabels(),
        fontfamily="monospace",
        rotation=45,
        ha="right",
        fontsize=fontsize
    )
    ax.set_yticklabels(
        ax.get_yticklabels(),
        fontfamily="monospace",
        rotation=0,
        fontsize=fontsize
    )
    
    plt.tight_layout()
    return fig, ax

In [ ]:
fig, ax = plot_global_normalized_fitness_heatmap(
    stats_df=amp_global_fitness_df,
    figsize=(5, 5),
    vmin=0,
    vmax=6,
    cmap="RdBu_r"
)
# plt.savefig(f"{base_path}/data/outputs/figures/summary-stats/heatmap_global_fitness_amp.png", dpi=1000, bbox_inches='tight', format="png")
plt.show()

In [ ]:
fig, ax = plot_global_normalized_fitness_heatmap(
    stats_df=azt_global_fitness_df,
    figsize=(5, 5),
    vmin=0,
    vmax=6,
    cmap="RdBu_r"
)
# plt.savefig(f"{base_path}/data/outputs/figures/summary-stats/heatmap_global_fitness_azt.png", dpi=1000, bbox_inches='tight', format="png")
plt.show()

In [ ]:
def plot_zoomed_global_normalized_fitness_heatmap(
    stats_df,
    max_mutations=2,
    height=8,
    vmin=0,
    vmax=6,
    cmap="PiYG",
    fontsize=10
):
    """
    Create a zoomed-in heatmap for global normalized fitness, focusing on genotypes 
    with low mutation counts.
    
    Parameters:
    -----------
    stats_df : polars.DataFrame
        DataFrame containing global fitness data with columns:
        'normalized_fitness', 'mutant_part1', 'mutant_part2', and 'mutant_profile'
    max_mutations : int, default=2
        Maximum number of mutations allowed for a genotype to be included
    height : float, default=8
        Height of the heatmap in inches
    vmin, vmax : float
        Min and max values for color scale
    cmap : str
        Colormap to use
    fontsize : int
        Font size for labels
    """
    import matplotlib.pyplot as plt
    from matplotlib import gridspec, colors
    import seaborn as sns
    import numpy as np
    import polars as pl

    # Helper function to count mutations
    def count_mutations(genotype):
        return sum(1 for c in genotype if c != '.')
    
    # Create the full pivot table
    pivot_df = stats_df.to_pandas().pivot_table(
        values="normalized_fitness",
        index="mutant_part2",
        columns="mutant_part1",
        aggfunc="mean"
    )
    
    # Filter and sort rows/columns by mutation count
    row_mutations = [(idx, count_mutations(idx)) for idx in pivot_df.index]
    col_mutations = [(col, count_mutations(col)) for col in pivot_df.columns]
    
    # Filter for low mutation counts
    filtered_rows = [row for row, count in row_mutations if count <= max_mutations]
    filtered_cols = [col for col, count in col_mutations if count <= max_mutations]
    
    # Sort filtered rows/cols by mutation count then alphabetically
    # Note: rows are sorted in reverse order to keep wildtype at bottom
    sorted_rows = sorted(filtered_rows, key=lambda x: (count_mutations(x), x), reverse=True)
    sorted_cols = sorted(filtered_cols, key=lambda x: (count_mutations(x), x))
    
    # Reindex the pivot table
    zoom_pivot_df = pivot_df.reindex(index=sorted_rows, columns=sorted_cols)
    
    # Get wildtype and dead mutant values
    wildtype = "............."
    dead_mutant = "XXXXXXXXXXXXX"
    
    wildtype_value = stats_df.filter(pl.col("mutant_profile") == wildtype)["normalized_fitness"].item()
    dead_value = stats_df.filter(pl.col("mutant_profile") == dead_mutant)["normalized_fitness"].item()
    
    # Calculate figure width to maintain square aspect ratio
    n_rows, n_cols = len(sorted_rows), len(sorted_cols)
    width = height * (n_cols / n_rows) * 1.3  # 1.3 factor for colorbar space
    
    # Create figure with GridSpec
    fig = plt.figure(figsize=(width, height))
    gs = gridspec.GridSpec(nrows=1, ncols=2, width_ratios=[1, 0.05], wspace=0.05)
    ax = fig.add_subplot(gs[0])
    cax = fig.add_subplot(gs[1])
    
    # Create the heatmap
    norm = colors.TwoSlopeNorm(vmin=vmin, vcenter=wildtype_value, vmax=vmax)
    
    heatmap = sns.heatmap(
        zoom_pivot_df,
        cmap=cmap,
        norm=norm,
        cbar=True,
        square=True,
        ax=ax,
        cbar_ax=cax,
        cbar_kws={"label": "Global Normalized Fitness"}
    )
    
    # Modify the color bar ticks and labels
    cbar = heatmap.collections[0].colorbar
    cbar.set_ticks([vmin, wildtype_value, vmax])
    cbar.set_ticklabels([
        f"{vmin:.1f}\n(Dead Mutant)",
        f"{wildtype_value:.1f}\n(Wildtype)",
        f"{vmax:.1f}"
    ])
    
    # Set labels
    ax.set_title("Zoomed Global Normalized Fitness", fontsize=fontsize)
    ax.set_xlabel("Mutant Part 1", fontsize=fontsize)
    ax.set_ylabel("Mutant Part 2", fontsize=fontsize)
    
    # Format tick labels
    ax.set_xticklabels(
        ax.get_xticklabels(),
        fontfamily="monospace",
        rotation=45,
        ha="right",
        fontsize=fontsize
    )
    ax.set_yticklabels(
        ax.get_yticklabels(),
        fontfamily="monospace",
        rotation=0,
        fontsize=fontsize
    )
    
    plt.tight_layout()
    return fig, ax

In [ ]:
fig, ax = plot_zoomed_global_normalized_fitness_heatmap(
    stats_df=amp_global_fitness_df,
    max_mutations=2,
    height=5,
    vmin=0,
    vmax=6,
    cmap="RdBu_r"
)
# plt.savefig(f"{base_path}/data/outputs/figures/summary-stats/zoomed_heatmap_global_fitness_amp.png", dpi=1000, bbox_inches='tight', format="png")
plt.show()

In [ ]:
fig, ax = plot_zoomed_global_normalized_fitness_heatmap(
    stats_df=azt_global_fitness_df,
    max_mutations=2,
    height=5,
    vmin=0,
    vmax=6,
    cmap="RdBu_r"
)
# plt.savefig(f"{base_path}/data/outputs/figures/summary-stats/zoomed_heatmap_global_fitness_azt.png", dpi=1000, bbox_inches='tight', format="png")
plt.show()

## Plot7: global fitness clustered by the number of mutations

In [ ]:
from scipy.stats import gaussian_kde

In [ ]:
def plot_fitness_vs_mutations(data, 
                            x_col='Epistatic Order',
                            y_col='Fitness',
                            figsize=(15, 4),
                            ylim=(0, 1.1),
                            title=None,
                            jitter_amount=0.1,
                            point_size=3,
                            point_alpha=0.2,
                            cmap='viridis',
                            save_path=None):
    """
    Create a scatter plot showing fitness vs number of mutations with density coloring.
    
    Parameters:
    -----------
    data : polars.DataFrame
        Input dataframe containing mutation and fitness data
    x_col : str, default='Epistatic Order'
        Column name for x-axis (number of mutations)
    y_col : str, default='Fitness'
        Column name for y-axis (fitness values)
    figsize : tuple, default=(15, 4)
        Figure dimensions (width, height)
    ylim : tuple, default=(0, 1.1)
        Y-axis limits (min, max)
    title : str, optional
        Plot title
    jitter_amount : float, default=0.1
        Amount of random jitter to add to x positions
    point_size : int, default=3
        Size of scatter points
    point_alpha : float, default=0.2
        Transparency of scatter points
    cmap : str, default='viridis'
        Colormap for density coloring
    save_path : str, optional
        If provided, save figure to this path
    
    Returns:
    --------
    matplotlib.figure.Figure
        The created figure
    """
    # Create figure and axis
    fig, ax = plt.subplots(figsize=figsize)
    
    # Add density-colored points for each category
    for order in data[x_col].unique():
        subset = data.filter(pl.col(x_col) == order)
        y = subset[y_col]
        
        # Add jitter only for orders greater than 0
        if order > 0:
            x = np.random.normal(order, jitter_amount, len(y))
        else:
            x = np.full(len(y), order)
        
        # Calculate density for coloring if we have multiple points
        if len(y) > 1:
            density = gaussian_kde(y)(y)
        else:
            density = np.array([0.5])
            
        # Scatter plot with density coloring
        scatter = ax.scatter(x, y, c=density, cmap=cmap,
                           s=point_size, alpha=point_alpha)
    
    # Add colorbar
    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label('Density')
    
    # Set labels and limits
    ax.set_ylabel(y_col)
    ax.set_xlabel('Number of Mutations')
    # ax.set_ylim(*ylim)
    # ax.set_xlim(-0.1, 13.3)
    ax.set_xticks(range(14))
    
    # Add grid
    ax.grid(True, alpha=0.3)
    
    # Set title if provided
    if title:
        ax.set_title(title)
    
    plt.tight_layout()
    
    # Save figure if path provided
    if save_path:
        plt.savefig(save_path, dpi=1000, bbox_inches='tight', format="png")
    
    return fig

In [ ]:
plot_fitness_vs_mutations(
    amp_global_epistasis_df,
    title='Fitness vs Number of Mutations for Ampicillin',
    cmap='crest_r',
    point_alpha=0.5,
    # save_path=f"{base_path}/data/outputs/figures/summary-stats/amp_fitness_vs_mutations.png"
)
plt.show()

In [ ]:
plot_fitness_vs_mutations(
    azt_global_epistasis_df,
    title='Fitness vs Number of Mutations for Aztreonam',
    cmap='crest_r',
    point_alpha=0.5,
    # save_path=f"{base_path}/data/outputs/figures/summary-stats/azt_fitness_vs_mutations.png"
)
plt.show()

## Plot8: global epistasis values clustered by the number of mutations

In [ ]:
plot_fitness_vs_mutations(
    amp_global_epistasis_df,
    title='Ensemble Averaging Epistasis vs Number of Mutations for Ampicillin',
    x_col='Epistatic Order',
    y_col='Ensemble Averaging',
    cmap='crest_r',
    point_alpha=0.5,
    # save_path=f"{base_path}/data/outputs/figures/summary-stats/amp_ensemble_averaging_vs_mutations.png"
)
plt.show()

In [ ]:
plot_fitness_vs_mutations(
    azt_global_epistasis_df,
    title='Ensemble Averaging Epistasis vs Number of Mutations for Aztreonam',
    x_col='Epistatic Order',
    y_col='Ensemble Averaging',
    cmap='crest_r',
    point_alpha=0.5,
    # save_path=f"{base_path}/data/outputs/figures/summary-stats/azt_ensemble_averaging_vs_mutations.png"
)
plt.show()

In [ ]:
plot_fitness_vs_mutations(
    amp_global_epistasis_df,
    title='Biochemical Definition Epistasis vs Number of Mutations for Ampicillin',
    x_col='Epistatic Order',
    y_col='Biochemical Definition',
    cmap='crest_r',
    point_alpha=0.5,
    # save_path=f"{base_path}/data/outputs/figures/summary-stats/amp_fitness_vs_mutations.png"
)
plt.show()

In [ ]:
plot_fitness_vs_mutations(
    azt_global_epistasis_df,
    title='Biochemical Definition Epistasis vs Number of Mutations for Aztreonam',
    x_col='Epistatic Order',
    y_col='Biochemical Definition',
    cmap='crest_r',
    point_alpha=0.5,
    # save_path=f"{base_path}/data/outputs/figures/summary-stats/amp_fitness_vs_mutations.png"
)
plt.show()

## Plot9: global epistasis values vs fitness

In [ ]:
def plot_epistasis_heatmap(df, drug_type, figsize=(15, 6)):
    """
    Create heatmaps showing the number of mutants and average epistatic order.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame containing the epistasis data
    drug_type : str
        Drug type ('AMP' or 'AZT') to determine appropriate bin ranges
    figsize : tuple, optional
        Figure size in inches (default: (15, 6))
    """
    # Set bin ranges based on drug type
    if drug_type == 'AMP':
        x_bins = np.arange(-0.15, 1.25, 0.1)
        y_bins = np.arange(-7.5, 7.5, 1)
    elif drug_type == 'AZT':
        x_bins = np.arange(-0.75, 6.75, 0.5)
        y_bins = np.arange(-32.5, 37.5, 5)
    else:
        raise ValueError("drug_type must be either 'AMP' or 'AZT'")

    # Filter data
    filtered_data = df[(df['Epistatic Order'] != 0) & 
                      (df['Epistatic Order'] != 13)]

    # Extract necessary columns
    x = filtered_data['Fitness'].to_numpy()
    y = filtered_data['Ensemble Averaging'].to_numpy()
    mutations = filtered_data['Epistatic Order'].to_numpy()

    # Create grid
    X, Y = np.meshgrid(x_bins, y_bins)

    # Calculate grid values
    count = np.zeros((len(y_bins) - 1, len(x_bins) - 1))
    mutation_sum = np.zeros((len(y_bins) - 1, len(x_bins) - 1))
    for i in range(len(x_bins) - 1):
        for j in range(len(y_bins) - 1):
            mask = (x >= x_bins[i]) & (x < x_bins[i + 1]) & \
                  (y >= y_bins[j]) & (y < y_bins[j + 1])
            count[j, i] = np.sum(mask)
            mutation_sum[j, i] = np.sum(mutations[mask])

    # Calculate log count and average mutations
    log_count = np.log10(count + 1)  # Add 1 to avoid log(0)
    avg_mutations = np.divide(mutation_sum, count, 
                            out=np.zeros_like(mutation_sum), 
                            where=count != 0)

    # Create figure and axes
    fig, axes = plt.subplots(1, 2, figsize=figsize)

    # Plot number of mutants heatmap
    cset1 = axes[0].pcolormesh(X, Y, log_count, cmap='Reds', 
                              edgecolors='black', linewidth=0.5, 
                              vmin=0, vmax=6)
    axes[0].set_xlabel('Fitness')
    axes[0].set_ylabel('Ensemble Averaging')
    fig.colorbar(cset1, ax=axes[0], label='log10(Number of Mutants + 1)')
    axes[0].set_title('Number of Mutants')

    # Annotate number of mutants
    for i in range(len(x_bins) - 1):
        for j in range(len(y_bins) - 1):
            if count[j, i] > 0:
                mean_val = count[j, i]
                x_pos = x_bins[i] + (x_bins[i+1] - x_bins[i])/2
                y_pos = y_bins[j] + (y_bins[j+1] - y_bins[j])/2
                axes[0].text(x_pos, y_pos, f'{mean_val:.0f}', 
                           color='black', ha='center', va='center', 
                           fontsize=8)

    # Plot average mutations heatmap
    cset2 = axes[1].pcolormesh(X, Y, avg_mutations, cmap='Blues', 
                              edgecolors='black', linewidth=0.5, 
                              vmin=0, vmax=15)
    axes[1].set_xlabel('Fitness')
    axes[1].set_ylabel('Ensemble Averaging')
    fig.colorbar(cset2, ax=axes[1], label='Average Epistatic order')
    axes[1].set_title('Average Number of Mutations')

    # Annotate average mutations
    for i in range(len(x_bins) - 1):
        for j in range(len(y_bins) - 1):
            if count[j, i] > 0:
                avg_val = avg_mutations[j, i]
                stdev_val = np.std(mutations[(x >= x_bins[i]) & 
                                           (x < x_bins[i + 1]) & 
                                           (y >= y_bins[j]) & 
                                           (y < y_bins[j + 1])])
                x_pos = x_bins[i] + (x_bins[i+1] - x_bins[i])/2
                y_pos = y_bins[j] + (y_bins[j+1] - y_bins[j])/2
                axes[1].text(x_pos, y_pos, f'{avg_val:.1f}\n±{stdev_val:.1f}', 
                           color='black', ha='center', va='center', 
                           fontsize=8)

    # Set x-ticks based on drug type
    if drug_type == 'AMP':
        for ax in axes:
            ax.set_xticks(np.arange(-0.1, 1.1, 0.1))
            ax.set_xticklabels([f'{tick:.1f}' for tick in np.arange(-0.1, 1.1, 0.1)])
    else:  # AZT
        for ax in axes:
            ax.set_xticks(np.arange(0, 7, 1))
            ax.set_xticklabels(np.arange(0, 7, 1))

    plt.tight_layout()
    return fig, axes

In [ ]:
fig, axes = plot_epistasis_heatmap(amp_global_epistasis_df.to_pandas(), 'AMP')
# plt.savefig(f"{base_path}/data/outputs/figures/summary-stats/global_epistasis_amp.png", dpi=1000, bbox_inches='tight', format="png")
plt.show()

In [ ]:
fig, axes = plot_epistasis_heatmap(azt_global_epistasis_df.to_pandas(), 'AZT')
# plt.savefig(f"{base_path}/data/outputs/figures/summary-stats/global_epistasis_azt.png", dpi=1000, bbox_inches='tight', format="png")
plt.show()

## Plot6: how we calculate local condition fitness and global fitness

In [ ]:
# Create figure with two subplots side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Plot distribution for Ampicillin
sns.histplot(data=amp_global_epistasis_df, 
             x='Fitness',
             bins=100,
             color='#E74C3C',
             alpha=0.6,
             ax=ax1)

ax1.set_title('Distribution of Global Fitness\nfor Ampicillin', fontsize=12)
ax1.set_xlabel('Fitness', fontsize=10)
ax1.set_ylabel('Count', fontsize=10)
ax1.grid(True, alpha=0.3)

# Plot distribution for Aztreonam 
sns.histplot(data=azt_global_epistasis_df,
             x='Fitness', 
             bins=100,
             color='#2E86C1',
             alpha=0.6,
             ax=ax2)

ax2.set_title('Distribution of Global Fitness\nfor Aztreonam', fontsize=12)
ax2.set_xlabel('Fitness', fontsize=10)
ax2.set_ylabel('Count', fontsize=10)
ax2.grid(True, alpha=0.3)

# Adjust layout and display
plt.tight_layout()
plt.show()


In [ ]:
# Find mutants with specific global fitness values in Ampicillin
target_fitness = [0.57, 0.8, 1.1]
selected_mutants = []

for target in target_fitness:
    # Convert Polars DataFrame to Pandas for this operation
    amp_df_pd = amp_global_fitness_df.to_pandas()
    
    # Find the mutant with fitness closest to target
    closest_idx = (amp_df_pd['normalized_fitness'] - target).abs().idxmin()
    closest_mutant = amp_df_pd.loc[closest_idx]
    selected_mutants.append(closest_mutant)

# Print the selected mutants and their fitness values
for mutant in selected_mutants:
    print(f"Mutant profile: {mutant['mutant_profile']}")
    print(f"Global fitness: {mutant['normalized_fitness']:.3f}")
    print()

In [ ]:
# add wildtype and dead mutant
selected = []
wildtype = "............."
dead = "XXXXXXXXXXXXX"
selected.append(wildtype)
selected.append(dead)

selected.extend([mutant['mutant_profile'] for mutant in selected_mutants])
selected


In [ ]:
df_amp = pd.read_csv(f"{REPO_DATA}/raw/Ampicillin_read_counts_per_genotype.csv", index_col=0)
df_azt = pd.read_csv(f"{REPO_DATA}/raw/Aztreonam_read_counts_per_genotype.csv", index_col=0)

df_amp_norm = df_amp.div(df_amp.sum())
df_azt_norm = df_azt.div(df_azt.sum())

meta = pd.read_csv(f"{REPO_DATA}/raw/metadata.csv")
meta = meta.set_index('Sample Name')
meta['Timepoint'] = meta['Timepoint'].str.strip('h').astype(int)

# Collect T0 sample for any given sample
def get_t0(sample):
    rep = meta.loc[sample,'Replicate']
    drug = meta.loc[sample, 'Drug']
    selected = meta[(meta['Drug']==drug) & (meta['Replicate']==rep) & (meta['Timepoint']==0)]
    if len(selected)>1:
        raise
    else:
        return selected.index.values[0]
    
def estimate_cell_count(df, meta):
    new_df = df.copy()
    for col in new_df.columns:
        od600 = meta.loc[col, 'OD600']
        # OD600=1 roughly equivalent to 5e8 E. coli cells
        new_df[col] = new_df[col] * od600 * 5e8 * 5
        new_df[col] = new_df[col].round().astype(int)
    return new_df

def cell_count_fold_change_from_T0(df, meta):
    new_df = df.copy(deep=True)
    
    for col in new_df.columns:
        new_df[col] = new_df[col].div(df[get_t0(col)])
    return new_df


df_amp_est_cell_count = estimate_cell_count(df_amp_norm, meta)
df_azt_est_cell_count = estimate_cell_count(df_azt_norm, meta)
df_amp_fc = cell_count_fold_change_from_T0(df_amp_est_cell_count, meta)
df_azt_fc = cell_count_fold_change_from_T0(df_azt_est_cell_count, meta)

In [ ]:
long_amp_fc = df_amp_fc.loc[selected,:].transpose().reset_index(names='Sample Name').melt(id_vars='Sample Name',value_name='fold_change')
long_amp_cc = df_amp_est_cell_count.loc[selected,:].transpose().reset_index(names='Sample Name').melt(id_vars='Sample Name',value_name='cell_count')
long_azt_fc = df_azt_fc.loc[selected,:].transpose().reset_index(names='Sample Name').melt(id_vars='Sample Name',value_name='fold_change')
long_azt_cc = df_azt_est_cell_count.loc[selected,:].transpose().reset_index(names='Sample Name').melt(id_vars='Sample Name',value_name='cell_count')

long_fc = pd.concat([long_amp_fc, long_azt_fc])
long_cc = pd.concat([long_amp_cc, long_azt_cc])
metap = meta.merge(long_fc, on='Sample Name', how='left')
metap = metap.merge(long_cc, on=['Sample Name','mut_profile_masked'], how='left')

In [ ]:
sns.pointplot(metap.query('Drug == "Ampicillin" & (Concentration==781 | `Sample Name`.str.contains("to"))'), x='Timepoint', y='fold_change', hue='mut_profile_masked',
              alpha=1, linewidth=1.5, dodge=False, legend=True)
# set y axis to log scale
plt.yscale('log')

# change legend '.............' to Wild Type, XXXXXXXXXXXXX to Dead Mutant. Align name with colors
handles, labels = plt.gca().get_legend_handles_labels()
labels = ['Wild Type' if label == '.............' else label for label in labels]
labels = ['Dead Mutant' if label == 'XXXXXXXXXXXXX' else label for label in labels]
plt.legend(handles, labels, title='Mutants', prop={'family': 'monospace'})

# change x axis label to Time (h)
plt.xlabel('Time (h)')

# change y axis label to Fold Change
plt.ylabel('Cell Density (AU)')

# plt.show()

# save the figure as a pdf
# plt.savefig(f"{base_path}/output/figures/basic-plots/time_traj_amp_781.pdf")


In [ ]:
# Get fitness data for selected mutants across all concentrations
selected_df = amp_long_df.filter(pl.col('mutant_profile').is_in(selected))

# Create a mapping dictionary for sorting order
sort_order = {mutant: idx for idx, mutant in enumerate(selected)}

# Add fitness column and sort by mutant_profile according to selected order
selected_df = (selected_df
    .with_columns(
        pl.struct(['replicate1', 'replicate2', 'replicate3'])
        .map_elements(lambda x: np.power(10, np.nanmedian([x['replicate1'], x['replicate2'], x['replicate3']])), 
                     return_dtype=pl.Float64)
        .alias('fitness')
    )
    .with_columns(
        pl.col('mutant_profile')
        .map_elements(lambda x: sort_order[x], return_dtype=pl.Int64)
        .alias('sort_idx')
    )
    .sort('sort_idx')
    .drop('sort_idx')
)

In [ ]:
selected_df

In [ ]:
# First get all unique mutants
all_mutants = list(amp_long_df['mutant_profile'].unique())

# Remove wild type and dead mutant from pool for random selection
wildtype = "............."
dead = "XXXXXXXXXXXXX"
background_mutants = [m for m in all_mutants if m not in [wildtype, dead]]

# Randomly sample 1000 mutants
np.random.seed(42)  # For reproducibility
sampled_mutants = np.random.choice(background_mutants, size=min(1000, len(background_mutants)), replace=False)

# Create the full selected list
selected = []
selected.append(wildtype)
selected.append(dead)
selected.extend(sampled_mutants)

df_amp = pd.read_csv(f"{REPO_DATA}/raw/Ampicillin_read_counts_per_genotype.csv", index_col=0)
df_azt = pd.read_csv(f"{REPO_DATA}/raw/Aztreonam_read_counts_per_genotype.csv", index_col=0)

df_amp_norm = df_amp.div(df_amp.sum())
df_azt_norm = df_azt.div(df_azt.sum())

meta = pd.read_csv(f"{REPO_DATA}/raw/metadata.csv")
meta = meta.set_index('Sample Name')
meta['Timepoint'] = meta['Timepoint'].str.strip('h').astype(int)

# Collect T0 sample for any given sample
def get_t0(sample):
    rep = meta.loc[sample,'Replicate']
    drug = meta.loc[sample, 'Drug']
    selected = meta[(meta['Drug']==drug) & (meta['Replicate']==rep) & (meta['Timepoint']==0)]
    if len(selected)>1:
        raise
    else:
        return selected.index.values[0]
    
def estimate_cell_count(df, meta):
    new_df = df.copy()
    for col in new_df.columns:
        od600 = meta.loc[col, 'OD600']
        # OD600=1 roughly equivalent to 5e8 E. coli cells
        new_df[col] = new_df[col] * od600 * 5e8 * 5
        new_df[col] = new_df[col].round().astype(int)
    return new_df

def cell_count_fold_change_from_T0(df, meta):
    new_df = df.copy(deep=True)
    
    for col in new_df.columns:
        new_df[col] = new_df[col].div(df[get_t0(col)])
    return new_df


df_amp_est_cell_count = estimate_cell_count(df_amp_norm, meta)
df_azt_est_cell_count = estimate_cell_count(df_azt_norm, meta)
df_amp_fc = cell_count_fold_change_from_T0(df_amp_est_cell_count, meta)
df_azt_fc = cell_count_fold_change_from_T0(df_azt_est_cell_count, meta)


long_amp_fc = df_amp_fc.loc[selected,:].transpose().reset_index(names='Sample Name').melt(id_vars='Sample Name',value_name='fold_change')
long_amp_cc = df_amp_est_cell_count.loc[selected,:].transpose().reset_index(names='Sample Name').melt(id_vars='Sample Name',value_name='cell_count')
long_azt_fc = df_azt_fc.loc[selected,:].transpose().reset_index(names='Sample Name').melt(id_vars='Sample Name',value_name='fold_change')
long_azt_cc = df_azt_est_cell_count.loc[selected,:].transpose().reset_index(names='Sample Name').melt(id_vars='Sample Name',value_name='cell_count')

long_fc = pd.concat([long_amp_fc, long_azt_fc])
long_cc = pd.concat([long_amp_cc, long_azt_cc])
metap = meta.merge(long_fc, on='Sample Name', how='left')
metap = metap.merge(long_cc, on=['Sample Name','mut_profile_masked'], how='left')

In [ ]:
# Create figure
plt.figure(figsize=FIGURE_SIZES['single_column'])

# Get data for specific concentration and timepoints
data = metap.query('Drug == "Ampicillin" & (Concentration==781 | `Sample Name`.str.contains("to"))')

# Plot background mutants in gray first
for mutant in sampled_mutants:
    mutant_data = data[data['mut_profile_masked'] == mutant]
    sns.pointplot(
        data=mutant_data, 
        x='Timepoint', 
        y='fold_change',
        color='0.7',  # Lighter gray
        alpha=0.3, 
        linewidth=0.5,
        errorbar=None, 
        linestyles='-',
        label='Other mutants' if mutant == sampled_mutants[0] else None  # Shorter label
    )

# Plot wild type and dead mutant on top
wt_data = data[data['mut_profile_masked'] == wildtype]
dead_data = data[data['mut_profile_masked'] == dead]

sns.pointplot(
    data=wt_data, 
    x='Timepoint', 
    y='fold_change',
    color='red', 
    linewidth=1, 
    label='Wild Type',
    errorbar=None, 
    linestyles='-', 
    markersize=3
)
sns.pointplot(
    data=dead_data, 
    x='Timepoint', 
    y='fold_change',
    color='black', 
    linewidth=1, 
    label='Dead Mutant', 
    errorbar=None, 
    linestyles='-', 
    markersize=3
)

# Customize plot
plt.yscale('log')
plt.xlabel('Time (h)')
plt.ylabel('Cell Density (AU)')
plt.title('Growth Trajectories in Ampicillin')

# Set x-axis to start from 0
plt.xlim(0, plt.xlim()[1])

# Create custom legend handles to match plot appearance
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='red', label='Wild Type', linewidth=1),
    Line2D([0], [0], color='black', label='Dead Mutant', linewidth=1),
    Line2D([0], [0], color='0.7', label='Other mutants', linewidth=1)
]

# Adjust legend with custom handles
plt.legend(
    handles=legend_elements,
    fontsize=6,  # Match title font size
    frameon=False,  # Remove legend frame
    title=None  # Remove legend title if unnecessary
)

# Reduce frame linewidth
plt.gca().spines['top'].set_linewidth(0.8)
plt.gca().spines['right'].set_linewidth(0.8)
plt.gca().spines['left'].set_linewidth(0.8)
plt.gca().spines['bottom'].set_linewidth(0.8)

# Tight layout for better spacing
plt.tight_layout()

# plt.savefig(f"{base_path}/data/outputs/figures/summary-stats/time_traj_amp_781_sampled1000.png", dpi=1000, bbox_inches='tight', format="png")
plt.show()

In [ ]:
# Create figure
plt.figure(figsize=FIGURE_SIZES['single_column'])

# Get data for specific concentration and timepoints
data = metap.query('Drug == "Aztreonam" & (Concentration==0.44 | `Sample Name`.str.contains("to"))')

# Plot background mutants in gray first
for mutant in sampled_mutants:
    mutant_data = data[data['mut_profile_masked'] == mutant]
    sns.pointplot(
        data=mutant_data, 
        x='Timepoint', 
        y='fold_change',
        color='0.7',  # Light gray
        alpha=0.3, 
        linewidth=0.5,
        errorbar=None, 
        linestyles='-',
        label='Other mutants' if mutant == sampled_mutants[0] else None
    )

# Plot wild type and dead mutant on top
wt_data = data[data['mut_profile_masked'] == wildtype]
dead_data = data[data['mut_profile_masked'] == dead]

sns.pointplot(
    data=wt_data, 
    x='Timepoint', 
    y='fold_change',
    color='red', 
    linewidth=1, 
    label='Wild Type',
    errorbar=None, 
    linestyles='-', 
    markersize=3
)
sns.pointplot(
    data=dead_data, 
    x='Timepoint', 
    y='fold_change',
    color='black', 
    linewidth=1, 
    label='Dead Mutant', 
    errorbar=None, 
    linestyles='-', 
    markersize=3
)

# Customize plot
plt.yscale('log')
plt.xlabel('Time (h)')
plt.ylabel('Cell Density (AU)')
plt.title('Growth Trajectories in Aztreonam')

# Set x-axis to start from 0
plt.xlim(0, plt.xlim()[1])

# Create custom legend handles
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='red', label='Wild Type', linewidth=1),
    Line2D([0], [0], color='black', label='Dead Mutant', linewidth=1),
    Line2D([0], [0], color='0.7', label='Other mutants', linewidth=1)
]

# Adjust legend with custom handles
plt.legend(
    handles=legend_elements,
    fontsize=6,
    frameon=False,
    title=None
)

# Reduce frame linewidth
plt.gca().spines['top'].set_linewidth(0.8)
plt.gca().spines['right'].set_linewidth(0.8)
plt.gca().spines['left'].set_linewidth(0.8)
plt.gca().spines['bottom'].set_linewidth(0.8)

# Tight layout for better spacing
plt.tight_layout()

# plt.savefig(f"{base_path}/data/outputs/figures/summary-stats/time_traj_azt_0.44_sampled1000.png", dpi=1000, bbox_inches='tight', format="png")
plt.show()